# 01 · Знакомство с данными
Что лежит в базе, сколько строк, нет ли дыр. Для демо-данных запусти Jupyter так: `FR_DB=data/demo.duckdb jupyter lab`.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import matplotlib.pyplot as plt
from storage.db import connect

def q(sql, params=()):
    """Запрос к базе -> DataFrame. Соединение только на чтение и сразу закрывается."""
    with connect(read_only=True) as con:
        return con.execute(sql, list(params)).df()

## Сколько строк в каждой таблице

In [ ]:
pd.concat([q(f"SELECT '{t}' AS tbl, count(*) AS n FROM {t}") for t in
           ["instruments", "funding", "open_interest", "klines", "spreads", "anomalies", "alerts", "collector_runs"]])

## Funding BTC за последний час (сдача шага 2)

In [ ]:
q("""SELECT ts, exchange, funding_rate * 100 AS pct_8h, raw_interval_h, mark_price
     FROM funding
     WHERE symbol = 'BTC' AND ts > (SELECT max(ts) FROM funding) - INTERVAL 1 HOUR
     ORDER BY ts DESC, exchange""")

## BTC сейчас на трёх биржах (сдача шага 3)
Три строки с одинаковым `ts` и сравнимыми ставками.

In [ ]:
q("""SELECT exchange, ts, funding_rate, raw_rate, raw_interval_h
     FROM funding WHERE symbol = 'BTC' AND ts = (SELECT max(ts) FROM funding) ORDER BY exchange""")

## Дубли и дыры
Дублей быть не может (первичный ключ), но проверить полезно. Дыры — слоты, где ответили не все биржи.

In [ ]:
q("""SELECT exchange, symbol, ts, count(*) AS n FROM funding GROUP BY ALL HAVING count(*) > 1""")

In [ ]:
q("""SELECT ts, count(DISTINCT exchange) AS exchanges, count(*) AS rows
     FROM funding GROUP BY ts HAVING count(DISTINCT exchange) < 3 ORDER BY ts DESC LIMIT 20""")

## Здоровье коллекторов (сдача шага 4)

In [ ]:
q("""SELECT collector, max(finished) AS last_run,
            count(*) FILTER (WHERE status = 'error') AS errors, count(*) AS runs
     FROM collector_runs GROUP BY collector ORDER BY collector""")

## Задание
1. У каких монет интервал funding не 8 часов? (таблица `instruments`)
2. Сколько МБ занимает файл базы и сколько строк прибавляется за сутки?
3. Найди самый длинный разрыв между соседними слотами в `funding` (оконная функция `lag`).